# OmniCare Clinical Copilot - Notebook 0: Setup & Models

This notebook sets up the environment, loads AI models, and initialises cloud services.

**Models:**
- `google/medgemma-1.5-4b-it` — Medical multimodal LLM (4-bit quantized)
- `openai/whisper-large-v3-turbo` — Speech-to-text ASR
- `google/hear-pytorch` — Health Acoustic Representations (cough/respiratory detection)

**Cloud Services:**
- Google Cloud Firestore — Persistent encounter/patient database
- Google Cloud Healthcare API — FHIR store + DICOM store
- HuggingFace Hub — Gated model access

**GPU Budget:** ~6-7 GB on T4 16GB (comfortable headroom)

## 0. Colab Bootstrap (run this first)

Auto-detects environment. In Colab, clones the private repo using your GitHub PAT.

**One-time setup:** In Colab, go to the **Key icon** in the left sidebar > Add a secret named `GITHUB_PAT` with your [GitHub Personal Access Token](https://github.com/settings/tokens).

In [ ]:
# ===========================================================
# Colab Bootstrap - run this cell first (works locally too)
# ===========================================================
import os, sys

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_DIR = '/content/omnicare-clinical-copilot'

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        token = userdata.get('GITHUB_PAT')
        repo_url = f'https://{token}@github.com/Yashground/omnicare-clinical-copilot.git'
        os.system(f'git clone {repo_url} {REPO_DIR}')
    NOTEBOOKS_DIR = os.path.join(REPO_DIR, 'notebooks')
    os.makedirs('/content/encounters', exist_ok=True)
    os.makedirs('/content/sample_data', exist_ok=True)
else:
    NOTEBOOKS_DIR = os.path.dirname(os.path.abspath('__file__'))

if NOTEBOOKS_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOKS_DIR)

print(f'Environment ready | Colab: {IN_COLAB} | Notebooks dir: {NOTEBOOKS_DIR}')

## 1. Install Dependencies

In [ ]:
%pip install -q transformers accelerate bitsandbytes torch
%pip install -q soundfile librosa
%pip install -q pydicom Pillow
%pip install -q fhir.resources pydantic
%pip install -q huggingface_hub
%pip install -q google-cloud-firestore
%pip install -q google-api-python-client google-auth

## 2. Authenticate with HuggingFace

MedGemma is a gated model — you must have accepted the license at
[huggingface.co/google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it).

In [ ]:
from huggingface_hub import login

# This will prompt for your HuggingFace token
# You can also set HF_TOKEN environment variable or use Colab secrets
login()

## 2b. GCP Authentication & Project Configuration

Authenticate with Google Cloud for Firestore and Healthcare API access.

In [ ]:
import os

# GCP Project Configuration
# Set your GCP project ID here (or use Colab secrets)
try:
    from google.colab import userdata
    GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
except Exception:
    GCP_PROJECT_ID = None

if not GCP_PROJECT_ID:
    GCP_PROJECT_ID = input("Enter your GCP Project ID: ").strip()

os.environ["GCP_PROJECT_ID"] = GCP_PROJECT_ID
os.environ["HEALTHCARE_LOCATION"] = "us-central1"
os.environ["HEALTHCARE_DATASET"] = "omnicare-dataset"

# Authenticate with GCP
try:
    from google.colab import auth
    auth.authenticate_user()
    print(f"GCP authenticated. Project: {GCP_PROJECT_ID}")
except ImportError:
    print("Not in Colab — using Application Default Credentials")

## 2c. Initialize Firestore Database

Connect to Firestore for persistent encounter/patient storage.

In [ ]:
# Initialize Firestore — this also verifies the connection
from utils.firestore_db import _get_db

try:
    db = _get_db()
    print(f"Firestore connected. Project: {db.project}")
    # Quick test: list existing encounters
    from utils.firestore_db import list_encounters
    existing = list_encounters()
    print(f"Existing encounters in Firestore: {len(existing)}")
except Exception as e:
    print(f"Firestore init failed: {e}")
    print("Falling back to local JSON storage (data will not persist across sessions).")

## 3. Check GPU Availability

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected. Models will run very slowly on CPU.")
    print("In Colab: Runtime -> Change runtime type -> T4 GPU")

## 2d. Setup Healthcare API (FHIR + DICOM Stores)

Create the FHIR store (for patient data, vitals, conditions) and DICOM store
(for medical images) in Google Cloud Healthcare API.

**Prerequisites:** Enable the Cloud Healthcare API in your GCP project:
```
gcloud services enable healthcare.googleapis.com
```

In [ ]:
# Enable Healthcare API (run once)
!gcloud services enable healthcare.googleapis.com --project=$GCP_PROJECT_ID

# Create FHIR + DICOM stores (idempotent — safe to re-run)
from utils.healthcare_api import setup_healthcare_stores

try:
    stores = setup_healthcare_stores()
    print(f"\nFHIR endpoint: {stores['fhir_base_url']}")
    print(f"DICOM endpoint: {stores['dicom_base_url']}")
except Exception as e:
    print(f"Healthcare API setup failed: {e}")
    print("FHIR/DICOM features will use local parsing mode.")

## 4. Load MedGemma 1.5-4b-it (4-bit Quantized)

This model handles:
- SOAP note generation from transcripts
- Admission note generation
- Radiology image analysis (multimodal)
- Discharge summary generation
- Clinical interpretation of HeAR findings

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MEDGEMMA_MODEL_ID = "google/medgemma-1.5-4b-it"

# 4-bit quantization config — fits in T4 16GB GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading {MEDGEMMA_MODEL_ID} with 4-bit quantization...")
medgemma_model = AutoModelForImageTextToText.from_pretrained(
    MEDGEMMA_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
medgemma_processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID)

print(f"MedGemma loaded. VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

## 5. Load Whisper Large v3 Turbo (ASR)

Used for medical audio transcription with vocabulary prompting.

In [ ]:
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

WHISPER_MODEL_ID = "openai/whisper-large-v3-turbo"

print(f"Loading {WHISPER_MODEL_ID}...")
whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to("cuda")

whisper_processor = AutoProcessor.from_pretrained(WHISPER_MODEL_ID)

asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model=whisper_model,
    tokenizer=whisper_processor.tokenizer,
    feature_extractor=whisper_processor.feature_extractor,
    torch_dtype=torch.float16,
    device="cuda"
)

print(f"Whisper loaded. Total VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

## 6. Load HeAR — Health Acoustic Representations (Cough Detection)

Google's HeAR model produces 512-dim embeddings from 2-second audio clips,
trained on 300M+ clips of coughing, breathing, throat clearing, laughing, and speaking.

Used by the HeAR Agent to detect respiratory events during consultations.

- **Model:** `google/hear-pytorch`
- **Input:** 2-second clips at 16 kHz mono
- **Output:** 512-dimensional health acoustic embeddings

In [ ]:
from utils.hear_helpers import load_hear_model, load_hear_preprocessor

# Load HeAR model
hear_model = load_hear_model(device="cuda")

# Load audio preprocessor (clones the HeAR repo for spectrogram conversion)
hear_preprocess = load_hear_preprocessor()

# Quick sanity check
import torch as _th
test_audio = _th.rand((1, 32000), dtype=_th.float32).to("cuda")
test_spec = hear_preprocess(test_audio)
with _th.no_grad():
    test_out = hear_model.forward(test_spec, return_dict=True, output_hidden_states=True)
embed_dim = test_out.last_hidden_state.shape[-1] if hasattr(test_out, 'last_hidden_state') else 'unknown'
print(f"HeAR sanity check passed. Embedding dim: {embed_dim}")
print(f"Total VRAM used: {_th.cuda.memory_allocated(0) / 1e9:.2f} GB")
del test_audio, test_spec, test_out

## 7. Initialize Clinical Orchestrator

Create the multi-agent orchestrator with all loaded models.
This will be used across all subsequent notebooks.

In [ ]:
from utils.agents import ClinicalOrchestrator

# Build the shared models dict for all agents
models = {
    "medgemma_model": medgemma_model,
    "medgemma_processor": medgemma_processor,
    "asr_pipeline": asr_pipeline,
    "hear_model": hear_model,
    "hear_preprocess": hear_preprocess,
    "device": "cuda",
}

# Create the orchestrator — used across all notebooks
orchestrator = ClinicalOrchestrator(models=models)

print("Clinical Orchestrator initialized with agents:")
for name, agent in orchestrator.agents.items():
    print(f"  - {agent.name}: {agent.role}")

## 8. Available Patient Scenarios

List the pre-built clinical scenarios for dynamic encounter generation.

In [ ]:
from utils.patient_simulator import list_scenarios

scenarios = list_scenarios()
print("Available clinical scenarios:")
for i, s in enumerate(scenarios):
    print(f"  [{i}] {s['id']}: {s['patient']} — {s['chief_complaint']}")

print(f"\nTotal: {len(scenarios)} scenarios")
print("Use get_scenario('scenario_id') or get_scenario() for random selection.")

## 9. GPU Memory Summary

In [ ]:
!nvidia-smi

---

**Setup complete!** All models loaded, cloud services initialized.

**Available in session:**
- `medgemma_model` / `medgemma_processor` — Medical LLM
- `asr_pipeline` — Whisper speech-to-text
- `hear_model` / `hear_preprocess` — HeAR cough detection
- `orchestrator` — Multi-agent clinical coordinator
- `models` — Shared models dict for agents

**Next:** Run `01_consultation_audio_soap.ipynb` for the consultation phase.

**Important:** Keep this notebook's runtime active — the models are loaded in GPU memory
and shared across notebooks via Colab's session state.